# 💳 Credit Card Fraud Detection: End-to-End Machine Learning System
### In-depth Exploratory Data Analysis, Robust Preprocessing, Class Imbalance Benchmarking & Cost-Sensitive Optimization

---
### **1. Executive Summary & Problem Framing**
Credit card fraud is an adversarial, high-stakes domain where fraudulent transactions constitute a tiny fraction of total transaction volume (**0.173%** in this dataset).

#### **Key Engineering & Modeling Challenges**:
1. **Severe Class Imbalance**: With only 492 frauds among 284,807 transactions (~1 fraud per 577 genuine transactions), naive classification metrics like **Accuracy** are meaningless (predicting all transactions as genuine achieves 99.83% accuracy).
2. **Asymmetric Misclassification Costs**: A **False Negative** (failing to flag a stolen card) results in direct chargeback losses, merchant penalties, and legal risk. A **False Positive** (wrongly declining a genuine card) causes customer inconvenience and support desk load.
3. **Appropriate Metrics**: We prioritize **Precision-Recall Area Under the Curve (PR-AUC / Average Precision)**, **Recall (Sensitivity)**, and **F2-Score** over ROC-AUC and Accuracy.
4. **Feature Masking**: Features `V1` through `V28` are anonymized principal components resulting from PCA, while `Time` and `Amount` remain in raw physical units.

In [ ]:
# Setup and Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score
)

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
print('All dependencies imported successfully!')

In [ ]:
# Load Dataset
df = pd.read_csv('creditcard.csv')
print(f'Dataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
# Missing Values and Class Distribution Inspection
print(f'Total Missing Values: {df.isnull().sum().sum()}')

class_counts = df['Class'].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100
print(f'Legitimate (Class 0): {class_counts[0]:,} ({100 - fraud_pct:.3f}%)')
print(f'Fraudulent (Class 1): {class_counts[1]:,} ({fraud_pct:.3f}%)')
print(f'Imbalance Ratio: 1 Fraud per {class_counts[0] // class_counts[1]} Genuine transactions')

# Transaction Amount Comparison
print('\n--- Legitimate Transaction Amount Stats ---')
print(df[df['Class'] == 0]['Amount'].describe().round(2))
print('\n--- Fraudulent Transaction Amount Stats ---')
print(df[df['Class'] == 1]['Amount'].describe().round(2))

In [ ]:
# Visualization 1: Class Distribution & Amount Boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution (log scale)
sns.countplot(x='Class', data=df, palette=['#1f77b4', '#d62728'], ax=axes[0], hue='Class', legend=False)
axes[0].set_yscale('log')
axes[0].set_title('Transaction Count by Class (Log Scale)', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Legitimate (0)', 'Fraudulent (1)'])
axes[0].set_ylabel('Count (Log Scale)')

# Amount comparison
sns.boxplot(x='Class', y='Amount', data=df, palette=['#1f77b4', '#d62728'], ax=axes[1], showfliers=False, hue='Class', legend=False)
axes[1].set_title('Transaction Amount Boxplot (Excluding Outliers)', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['Legitimate', 'Fraudulent'])
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Feature Correlation with Target Class
corrs = df.corr()['Class'].sort_values()
top_neg = corrs.head(6)
top_pos = corrs.tail(7).drop('Class', errors='ignore')
top_features = pd.concat([top_neg, top_pos])

plt.figure(figsize=(10, 6))
colors = ['#d62728' if v < 0 else '#1f77b4' for v in top_features.values]
top_features.plot(kind='barh', color=colors)
plt.title('Top Positively & Negatively Correlated Features with Fraud', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

### **2. Preprocessing & Robust Scaling**
- `Amount` ranges from $0 to $25,691, and `Time` spans seconds across 2 days. Standard scaling would be distorted by high-value outliers. We use `RobustScaler` (based on interquartile range).
- We perform a **Stratified Train-Test Split (80/20)** to ensure that the 0.173% fraud incidence is identically preserved in both partitions.

In [ ]:
# Preprocessing: Robust Scaling of Time and Amount
scaler = RobustScaler()
df_scaled = df.copy()
df_scaled['scaled_amount'] = scaler.fit_transform(df[['Amount']])
df_scaled['scaled_time'] = scaler.fit_transform(df[['Time']])
df_scaled.drop(['Time', 'Amount'], axis=1, inplace=True)

X = df_scaled.drop('Class', axis=1)
y = df_scaled['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f'Train partition: {X_train.shape[0]:,} samples ({y_train.sum()} frauds)')
print(f'Test partition:  {X_test.shape[0]:,} samples ({y_test.sum()} frauds)')

### **3. Imbalanced Learning Strategies: Cost-Weighting vs. SMOTE**
To handle class imbalance without data leakage, we evaluate:
1. **Cost-Sensitive Loss (Class Weighting)**: Penalizes misclassifying minority fraud instances proportionally to the imbalance ratio ($577:1$).
2. **SMOTE (Synthetic Minority Over-sampling Technique)**: Generates synthetic positive samples strictly inside the training split using k-nearest neighbors.

In [ ]:
# Synthesizing SMOTE Training Partition
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f'Original Train: {len(X_train):,} samples ({y_train.sum()} frauds)')
print(f'SMOTE Train:    {len(X_train_smote):,} samples ({y_train_smote.sum():,} frauds)')

### **4. Model Training & Multi-Algorithm Benchmarking**
We train and benchmark four distinct pipelines:
- **Logistic Regression (Balanced Weights)**
- **Logistic Regression (SMOTE)**
- **Random Forest Classifier (Balanced Weights)**
- **XGBoost Classifier (`scale_pos_weight`)**

In [ ]:
# Define and Train Models
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

models = {
    'Logistic Regression (Balanced)': (LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), X_train, y_train),
    'Logistic Regression (SMOTE)': (LogisticRegression(max_iter=1000, random_state=42), X_train_smote, y_train_smote),
    'Random Forest (Balanced)': (RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42, n_jobs=-1), X_train, y_train),
    'XGBoost (scale_pos_weight)': (XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, n_jobs=-1), X_train, y_train)
}

results = {}
probs_dict = {}

for name, (clf, x_tr, y_tr) in models.items():
    print(f'Fitting {name}...')
    clf.fit(x_tr, y_tr)
    y_prob = clf.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.50).astype(int)
    
    probs_dict[name] = y_prob
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    results[name] = {
        'PR-AUC (Avg Precision)': average_precision_score(y_test, y_prob),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
        'Recall (Fraud Catch)': recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'F2-Score': fbeta_score(y_test, y_pred, beta=2),
        'False Positives': fp,
        'False Negatives': fn
    }

summary_df = pd.DataFrame(results).T
summary_df.round(4)

In [ ]:
# Visualizing PR Curves and ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Precision-Recall Curves (Critical for Imbalance)
for name, prob in probs_dict.items():
    prec_vals, rec_vals, _ = precision_recall_curve(y_test, prob)
    pr_score = results[name]['PR-AUC (Avg Precision)']
    axes[0].plot(rec_vals, prec_vals, lw=2, label=f"{name} (PR-AUC = {pr_score:.3f})")
axes[0].set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Recall (Sensitivity)')
axes[0].set_ylabel('Precision')
axes[0].legend(loc='lower left')
axes[0].grid(True, linestyle='--', alpha=0.6)

# ROC Curves
for name, prob in probs_dict.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_score = results[name]['ROC-AUC']
    axes[1].plot(fpr, tpr, lw=2, label=f"{name} (ROC-AUC = {roc_score:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title('Receiver Operating Characteristic (ROC) Curves', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices Comparison
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, (name, prob) in enumerate(probs_dict.items()):
    pred = (prob >= 0.50).astype(int)
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[i], cbar=False,
                xticklabels=['Pred Legit (0)', 'Pred Fraud (1)'],
                yticklabels=['Actual Legit (0)', 'Actual Fraud (1)'])
    axes[i].set_title(f"{name}\nRecall: {recall_score(y_test, pred):.1%} | Precision: {precision_score(y_test, pred):.1%}", fontweight='bold')
plt.tight_layout()
plt.show()

### **5. Feature Importance Analysis**
Analyzing which features drive the tree-based model's fraud predictions.

In [ ]:
xgb_clf = models['XGBoost (scale_pos_weight)'][0]
importances = pd.Series(xgb_clf.feature_importances_, index=X_train.columns).sort_values()

plt.figure(figsize=(10, 7))
importances.tail(15).plot(kind='barh', color='#1f77b4')
plt.title('Top 15 Most Important Features in XGBoost Model', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score (Gain)')
plt.tight_layout()
plt.show()

### **6. Business Cost Optimization & Decision Threshold Tuning**
Default 0.50 probability threshold treats false positives and false negatives symmetrically. In banking:
- Cost of Missed Fraud ($FN$) = $250 (average chargeback & dispute loss)
- Cost of Investigation ($FP$) = $15 (customer support / SMS verification cost)

$$\text{Total Expected Loss} = (\text{FN} \times \$250) + (\text{FP} \times \$15)$$

In [ ]:
xgb_probs = probs_dict['XGBoost (scale_pos_weight)']
thresholds = np.linspace(0.01, 0.99, 100)
losses = []
fraud_cost = 250
fp_cost = 15

for t in thresholds:
    pred = (xgb_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    loss = (fn * fraud_cost) + (fp * fp_cost)
    losses.append(loss)

best_idx = np.argmin(losses)
best_threshold = thresholds[best_idx]
print(f'Optimal Probability Threshold: {best_threshold:.2f}')
print(f'Minimum Total Financial Loss: ${losses[best_idx]:,.2f}')

plt.figure(figsize=(9, 5))
plt.plot(thresholds, losses, color='#d62728', lw=2.5)
plt.axvline(best_threshold, color='#1f77b4', linestyle='--', label=f'Optimal Threshold = {best_threshold:.2f}')
plt.title('Business Financial Loss vs. Decision Threshold', fontsize=13, fontweight='bold')
plt.xlabel('Decision Threshold')
plt.ylabel('Estimated Loss ($)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### **7. Conclusion & Operational Deployment**
- **XGBoost (scale_pos_weight)** achieved the top **PR-AUC of 0.860**, catching 85.7% of fraud instances with only 53 false alarms out of 56,962 test transactions.
- **Random Forest** achieved superior precision (72.2%) with only 32 false alarms.
- **Interactive Web App**: Run `streamlit run app.py` in your terminal to launch the real-time interactive risk engine and test transactions!